# 🥉 Bronze Layer Pipeline
**الهدف:** تحميل البيانات الخام من ملف CSV وإضافة Metadata وحفظ النتيجة كـ Bronze output.

---

## 📦 Cell 1 — الاستيرادات والمكتبات

In [1]:
import pandas as pd
import os
from datetime import datetime

## ⚙️ Cell 2 — الإعدادات والمتغيرات الثابتة
غيّر اسم الملف من هنا لو احتجت.

In [2]:
SOURCE_FILE = "waqi-covid19-airqualitydata-2023Q4-1.csv"
OUTPUT_FILE = "bronze_output.csv"
SOURCE_FILE_NAME = "waqi-covid19-airqualitydata-2023Q4-1.csv"

## 📂 Cell 3 — دالة تحميل البيانات الخام
بتفتح الـ CSV وتحمله كـ DataFrame — كل القيم بتتقرا كـ `str` عشان منضيعش بيانات.

In [3]:
def load_raw_data(filepath: str) -> pd.DataFrame:
    print(f"[BRONZE] Loading raw data from: {filepath}")
    try:
        df = pd.read_csv(
            filepath,
            dtype=str,
            keep_default_na=False,
            encoding="utf-8",
        )
        print(f"[BRONZE] ✓ Loaded {len(df):,} rows × {len(df.columns)} columns")
        return df
    except FileNotFoundError:
        print(f"[BRONZE] ✗ ERROR: File not found at path: {filepath}")
        raise
    except pd.errors.EmptyDataError:
        print("[BRONZE] ✗ ERROR: The source file is empty.")
        raise
    except Exception as e:
        print(f"[BRONZE] ✗ UNEXPECTED ERROR while loading data: {e}")
        raise

## 🏷️ Cell 4 — دالة إضافة الميتاداتا
بتضيف عمودين: `ingestion_date` (وقت التحميل) و `source_file` (اسم الملف الأصلي).

In [4]:
def add_metadata_columns(df: pd.DataFrame, source_file_name: str) -> pd.DataFrame:
    print("[BRONZE] Adding metadata columns: ingestion_date, source_file ...")
    try:
        ingestion_timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
        df = df.copy()
        df["ingestion_date"] = ingestion_timestamp
        df["source_file"] = source_file_name
        print(f"[BRONZE] ✓ ingestion_date set to: {ingestion_timestamp}")
        print(f"[BRONZE] ✓ source_file set to   : {source_file_name}")
        return df
    except Exception as e:
        print(f"[BRONZE] ✗ ERROR while adding metadata: {e}")
        raise

## 💾 Cell 5 — دالة حفظ ملف الإخراج
بتحفظ الـ DataFrame كـ CSV وبتطبع عدد الصفوف والأعمدة وحجم الملف.

In [5]:
def save_bronze_output(df: pd.DataFrame, output_path: str) -> None:
    print(f"[BRONZE] Saving bronze output to: {output_path}")
    try:
        directory = os.path.dirname(output_path)
        if directory:
            os.makedirs(directory, exist_ok=True)
        df.to_csv(output_path, index=False, encoding="utf-8")
        file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"[BRONZE] ✓ File saved successfully.")
        print(f"[BRONZE]   → Rows    : {len(df):,}")
        print(f"[BRONZE]   → Columns : {len(df.columns)}")
        print(f"[BRONZE]   → Size    : {file_size_mb:.2f} MB")
    except PermissionError:
        print(f"[BRONZE] ✗ ERROR: No write permission at: {output_path}")
        raise
    except Exception as e:
        print(f"[BRONZE] ✗ ERROR while saving bronze output: {e}")
        raise

## 📊 Cell 6 — دالة طباعة ملخص التنفيذ
بتطبع تقرير موجز: عدد الصفوف، الأعمدة، التكرارات، والقيم الفاضية.

In [6]:
def print_bronze_summary(df: pd.DataFrame) -> None:
    print("\n" + "=" * 60)
    print("  BRONZE LAYER — EXECUTION SUMMARY")
    print("=" * 60)
    print(f"  Total Records   : {len(df):,}")
    print(f"  Total Columns   : {len(df.columns)}")
    print(f"  Column Names    : {df.columns.tolist()}")
    print(f"  Duplicate Rows  : {df.duplicated().sum():,}")
    empty_counts = (df == "").sum()
    total_empty = empty_counts.sum()
    print(f"  Empty Values    : {total_empty:,}")
    print(f"  Ingestion Date  : {df['ingestion_date'].iloc[0]}")
    print(f"  Source File     : {df['source_file'].iloc[0]}")
    print("=" * 60 + "\n")

## ▶️ Cell 7 — تشغيل الـ Pipeline
شغّل الـ Pipeline كله من هنا — بيستدعي الدوال الأربعة بالترتيب.

In [7]:
def run_bronze_pipeline():
    print("\n" + "=" * 60)
    print("  BRONZE LAYER PIPELINE — START")
    print("=" * 60 + "\n")
    try:
        df_raw = load_raw_data(SOURCE_FILE)
        df_bronze = add_metadata_columns(df_raw, SOURCE_FILE_NAME)
        save_bronze_output(df_bronze, OUTPUT_FILE)
        print_bronze_summary(df_bronze)
        print("[BRONZE] ✓ Pipeline completed successfully.\n")
        return df_bronze
    except Exception as e:
        print(f"\n[BRONZE] ✗ PIPELINE FAILED: {e}")
        raise

df_bronze = run_bronze_pipeline()


  BRONZE LAYER PIPELINE — START

[BRONZE] Loading raw data from: waqi-covid19-airqualitydata-2023Q4-1.csv
[BRONZE] ✓ Loaded 217,731 rows × 9 columns
[BRONZE] Adding metadata columns: ingestion_date, source_file ...
[BRONZE] ✓ ingestion_date set to: 2026-06-24 16:11:21 UTC
[BRONZE] ✓ source_file set to   : waqi-covid19-airqualitydata-2023Q4-1.csv
[BRONZE] Saving bronze output to: bronze_output.csv


C:\Users\Omar Fathy\AppData\Local\Temp\ipykernel_4248\1662462814.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ingestion_timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")


[BRONZE] ✓ File saved successfully.
[BRONZE]   → Rows    : 217,731
[BRONZE]   → Columns : 11
[BRONZE]   → Size    : 24.23 MB

  BRONZE LAYER — EXECUTION SUMMARY
  Total Records   : 217,731
  Total Columns   : 11
  Column Names    : ['Date', 'Country', 'City', 'Specie', 'count', 'min', 'max', 'median', 'variance', 'ingestion_date', 'source_file']
  Duplicate Rows  : 0
  Empty Values    : 0
  Ingestion Date  : 2026-06-24 16:11:21 UTC
  Source File     : waqi-covid19-airqualitydata-2023Q4-1.csv

[BRONZE] ✓ Pipeline completed successfully.

